In [79]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

# Preparation of Data Frames

In [80]:
# Reading the saved data pickle file
df  = pd.DataFrame()
# df = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate.pkl', compression=None)
# df1 = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/pickled_filtered_lead_para_sp500_clean.pkl', compression=None)


df=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticles.pkl', compression=None)
df_time=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticlesTime.pkl', compression=None)

df.index= pd.to_datetime(df.index)
df.Date= pd.to_datetime(df.Date)
df_vix=pd.read_csv('/Users/Farid Srouji/Documents/Research/master/globaldata/vix.csv')
df_vix = df_vix.fillna(0)
# df_vix["Date"] = df_vix.index
# df_vix.index= pd.to_datetime(df_vix.index)
df_vix.Date= pd.to_datetime(df_vix.Date)
# df_vix.index= pd.to_datetime(df_vix.index)


# del df['Date']
# del df_vix['Date']

df_vix_2010_2021=pd.read_csv('/Users/Farid Srouji/Documents/Research/master/globaldata/vix_2010_2021.csv')


In [99]:
print(df_time["PressReleaseCount"].loc["2019-09-25":"2020-01-31"].apply(np.int).sum())
print(df_time["PressReleaseCount"].loc["2020-02-01":"2020-05-31"].apply(np.int).sum())
print(df_time["PressReleaseCount"].loc["2020-06-01":"2021-08-13"].apply(np.int).sum())

6503
3397


In [ ]:
df_vix

In [ ]:
df = pd.merge(df_vix,df,how='outer', on='Date')
df=df.fillna(0)

# S&P 500

In [ ]:
#if you get an error after executing the code, try adding below:
pd.core.common.is_list_like = pd.api.types.is_list_like

import pandas_datareader.data as web
import datetime

start = datetime.datetime(2019, 9, 25)
end = datetime.datetime(2021, 8, 13)
SP500  = pd.DataFrame()

SP500 = web.DataReader(['sp500'], 'fred', start, end)

In [ ]:
SP500 = SP500.reset_index()

In [ ]:
SP500

In [ ]:
df = df.join(SP500)

In [ ]:
sp500_yesterday = df["sp500"].shift(periods=1)
df["sp500_ln_return"]=np.log(df["sp500"]/sp500_yesterday)

In [ ]:
df.index = df.Date

In [ ]:
df["vix"] = df["Adj Close"]
vix_yesterday = df["vix"].shift(periods=1)
df["vix_ln_return"]=np.log(df["vix"]/vix_yesterday)
df['sentiment'] = df[["sentiment"]].apply(pd.to_numeric)
df["sentiment_ewm"]=df["sentiment"].ewm(10).mean()


In [ ]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['year'] = [d.year for d in df.index]
df['month'] = [d.strftime('%b') for d in df.index]
df['day'] = [d for d in df.index.day_name()]
years = df['year'].unique()
df['month-year'] = [d.strftime('%b') + str(d.year) for d in df.index]

In [ ]:
df_part1 = df.loc['2019-09-25':'2020-02-14']
df_part2 = df.loc['2020-02-15':'2020-05-31']
df_part3 = df.loc['2020-06-01':'2021-08-13']

In [ ]:
df

# VIX 2010 - 2021

In [ ]:
# Import Data
df_vix_2010_2021=pd.read_csv('/Users/Farid Srouji/Documents/Research/master/globaldata/vix_2010_2021.csv')
df_vix_2010_2021['Date']= pd.to_datetime(df_vix_2010_2021['Date'])
df_vix_2010_2021.index= df_vix_2010_2021['Date']
print("df_vix_2010_2021['Date']="+str(type(df_vix_2010_2021['Date'])))
print("df_part3['Date']="+str(type(df_part3['Date'])))

df_vix_2010_2021["vix"] = df_vix_2010_2021["Adj Close"]#.apply(np.int)
df_vix_2010_2021["vix_ln_return"]=np.log(df_vix_2010_2021["vix"]/df_vix_2010_2021["vix"].shift(1))

import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()] 

df_vix_2010_2021['year'] = [d.year for d in df_vix_2010_2021.Date]
df_vix_2010_2021['month'] = [d.strftime('%b') for d in df_vix_2010_2021.Date]
df_vix_2010_2021['day'] = [d for d in df_vix_2010_2021.index.day_name()]
years = df_vix_2010_2021['year'].unique()
df_vix_2010_2021['month-year'] = [d.strftime('%b') + str(d.year) for d in df_vix_2010_2021.Date]

In [ ]:
df_part3.fillna(0, inplace=True)
